# Validar o Iterated Greedy em Java e Python

Este notebook verifica se Java e Python devolvem **o mesmo conjunto de vértices**,
com a mesma semente, o mesmo grafo e o mesmo orçamento de iterações. A verificação
também testa se cada conjunto domina o grafo. Não realiza comparação de desempenho.

Coloque `iterated_greedy.py` em `src/complex_networks/algorithms/mds/` e este notebook
em `notebooks/`. Use o kernel do projeto, com pandas e NetworkX, e um JDK 11 ou superior.
A função `mds_solver_iterated_greedy()` é importada do módulo existente; as células
não alteram esse arquivo nem são necessárias para usá-lo em outros experimentos.

[Artigo](https://doi.org/10.1016/j.matcom.2022.12.018) ·
[Java dos autores](https://github.com/AlejandraCasado/MinimumDominatingSet)


## 1. Configuração

Escolha `INSTANCE_NAME` e `BETA`. Os pares `(semente, orçamento)` e o modo individual
ou em lote são configurados na célula **Executar a validação**.

Uma iteração conta destruição, reconstrução e busca local; orçamento `0` verifica
a solução inicial após a busca local. As versões podem terminar antes do orçamento
quando não há vértices móveis. O número de iterações serve aqui para alinhar os
passos das duas implementações, não para medir sua velocidade.


In [1]:
import gc
import shutil
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from operator import index

import pandas as pd
from IPython.display import display

cwd = Path.cwd()
for directory in (cwd, *cwd.parents):
    src = directory / "src"
    if (src / "complex_networks").is_dir():
        for import_path in (directory, src):
            if str(import_path) not in sys.path:
                sys.path.insert(0, str(import_path))
        break

from src.complex_networks.algorithms.mds.iterated_greedy import mds_solver_iterated_greedy
from src.complex_networks.config import PROJECT_ROOT, RAW_DATA_DIR
from src.complex_networks.io import load_edgelist, preprocess
from src.complex_networks.utils import iter_files

JAVA_REPO = Path(PROJECT_ROOT) / "libs" / "MinimumDominatingSet"
JAVA_URL = "https://github.com/AlejandraCasado/MinimumDominatingSet.git"
JAVA_REVISION = "d06cdadaaf3e303ce909b1fec93b90a17ee785b1"

INSTANCE_NAME = "amazon0302"
BETA = 0.2
ORDEM_ARQUIVOS = "size"


## 2. Preparar a referência Java

O notebook usa `JAVA_REPO` quando disponível. Caso contrário, obtém a revisão
`JAVA_REVISION` em um diretório temporário. A compilação e a execução são feitas
pelas células Python, sem comandos de terminal.

A referência temporária usa listas de adjacência, acrescenta a execução por
orçamento fixo e devolve os IDs do melhor conjunto. O repositório local é preservado.

**Escopo da validação:** a referência é o modo **canônico** usado no porte anterior.
Ele ordena os IDs antes dos embaralhamentos e fixa desempates que, no Java publicado,
dependem da iteração dos `HashSet`. Os sorteios seguem `java.util.Random`.
As varreduras são aceleradas com estruturas incrementais; a destruição é limitada
aos vértices móveis disponíveis, como no Python. O teste verifica a igualdade com
essa referência; não demonstra igualdade para toda ordem possível dos `HashSet`
do Java publicado nem constitui uma prova para todas as instâncias.


In [2]:
JAVA_COMPILER = """
import javax.tools.JavaCompiler;
import javax.tools.ToolProvider;

class CompileIg {
    public static void main(String[] args) {
        JavaCompiler compiler = ToolProvider.getSystemJavaCompiler();
        if (compiler == null) {
            throw new IllegalStateException("O JDK precisa disponibilizar o compilador Java.");
        }
        System.exit(compiler.run(null, System.out, System.err, args));
    }
}
"""

JAVA_MAIN = """
import heuristic.algorithm.IG;
import heuristic.constructive.GIP;
import heuristic.localSearch.LocalSearchEfficient_1_1;
import heuristic.structure.Instance;
import heuristic.structure.RandomManager;

import java.io.BufferedReader;
import java.io.PrintWriter;
import java.nio.file.Files;
import java.nio.file.Path;
import java.util.*;

public class IgFixedMain {
    @SuppressWarnings("unchecked")
    public static void main(String[] args) throws Exception {
        Instance graph;
        try (BufferedReader input = Files.newBufferedReader(Path.of(args[0]))) {
            StringTokenizer header = new StringTokenizer(input.readLine());
            int n = Integer.parseInt(header.nextToken());
            int m = Integer.parseInt(header.nextToken());
            List<Integer>[] adjacency = (List<Integer>[]) new List<?>[n];
            Set<Integer> vertices = new HashSet<>();
            for (int i = 0; i < n; i++) {
                adjacency[i] = new ArrayList<>();
                vertices.add(i);
            }
            for (int i = 0; i < m; i++) {
                StringTokenizer edge = new StringTokenizer(input.readLine());
                int u = Integer.parseInt(edge.nextToken());
                int v = Integer.parseInt(edge.nextToken());
                adjacency[u].add(v);
                adjacency[v].add(u);
            }
            for (List<Integer> row : adjacency) {
                Collections.sort(row);
            }
            graph = new Instance(n, adjacency, vertices);
        }

        RandomManager.setSeed(Long.parseLong(args[1]));
        IG algorithm = new IG(new GIP(), new LocalSearchEfficient_1_1());
        Set<Integer> solution = algorithm.executeFixed(
            graph, Integer.parseInt(args[2]), Float.parseFloat(args[3])
        );
        PrintWriter output = new PrintWriter(System.out);
        output.print("IG_SOLUTION " + solution.size());
        solution.stream().sorted().forEach(vertex -> output.print(" " + vertex));
        output.println();
        output.flush();
    }
}
"""

JAVA_FIXED_METHOD = """
    public java.util.Set<Integer> executeFixed(Instance instance, int iterations, float beta) {
        Solution solution = firstSol(instance);
        java.util.Set<Integer> best = new java.util.HashSet<>(solution.getSelectedNodes());
        int count = (int)Math.ceil(beta * solution.getSelectedNotSupportNodes().size());
        for (int i = 0; i < iterations && !solution.getSelectedNotSupportNodes().isEmpty(); i++) {
            destruct(solution, Math.min(count, solution.getSelectedNotSupportNodes().size()));
            construct(solution);
            executeLocalSearch(solution, instance);
            if (solution.evaluate() < best.size()) {
                best = new java.util.HashSet<>(solution.getSelectedNodes());
            }
        }
        return best;
    }
"""

JAVA_CACHE_MEMBERS = """
    private final TreeSet<Integer> redundant = new TreeSet<>();
    private int[] privateCounts;
    private PriorityQueue<Long> candidates;
    private int[] gains;

    private void changePrivate(int node, int delta) {
        privateCounts[node] += delta;
        if (privateCounts[node] == 0 && selectedNotSupportNodes.contains(node)) {
            redundant.add(node);
        } else {
            redundant.remove(node);
        }
    }

    private void updateWatcher(int node, int owner, boolean adding) {
        Set<Integer> seen = watchers[node];
        if (seen.size() == 1) changePrivate(seen.iterator().next(), -1);
        if (adding) {
            if (seen.isEmpty()) {
                numCovered++;
                if (candidates != null) {
                    for (int candidate : instance.getAdyacencyList()[node]) {
                        if (unSelectedNodes.contains(candidate)) {
                            gains[candidate]--;
                            candidates.add(greedyKey(candidate));
                        }
                    }
                }
            }
            seen.add(owner);
        } else {
            seen.remove(owner);
            if (seen.isEmpty()) numCovered--;
        }
        if (seen.size() == 1) changePrivate(seen.iterator().next(), 1);
    }

    private long greedyKey(int node) {
        return ((long)-gains[node] << 32) | node;
    }

    public int getBestNextNodeWithQueue() {
        if (candidates == null) {
            gains = new int[instance.getNumNodes()];
            List<Long> entries = new ArrayList<>(unSelectedNodes.size());
            for (int node : unSelectedNodes) {
                if (instance.getLeavesNodes().contains(node)) continue;
                for (int neighbor : instance.getAdyacencyList()[node]) {
                    if (watchers[neighbor].isEmpty()) gains[node]++;
                }
                entries.add(greedyKey(node));
            }
            candidates = new PriorityQueue<>(entries);
        }
        while (!candidates.isEmpty()) {
            long entry = candidates.remove();
            int node = (int)entry;
            if (unSelectedNodes.contains(node) && entry == greedyKey(node)) return node;
        }
        return -1;
    }
"""

JAVA_ADD_WATCHER = """
    private void addWatcher(int selectedNode){
        updateWatcher(selectedNode, selectedNode, true);
        for (int neighbor : instance.getAdyacencyList()[selectedNode]) {
            updateWatcher(neighbor, selectedNode, true);
        }
        if (isFeasible()) candidates = null;
    }

"""

JAVA_REMOVE_WATCHER = """
    private void removeWatcher(int selectedNode){
        candidates = null;
        updateWatcher(selectedNode, selectedNode, false);
        for (int neighbor : instance.getAdyacencyList()[selectedNode]) {
            updateWatcher(neighbor, selectedNode, false);
        }
    }

"""

In [3]:
def run_command(args):
    completed = subprocess.run(args, capture_output=True, text=True, encoding="utf-8")
    if completed.returncode:
        message = completed.stderr.strip() or completed.stdout.strip()
        raise RuntimeError(f"{Path(args[0]).name} saiu com código {completed.returncode}. {message}")
    return completed.stdout


def java_checkout(repo, directory):
    if repo.is_dir():
        return repo / "Code" / "src" / "heuristic"

    checkout = directory / "upstream"
    run_command(["git", "init", "--quiet", str(checkout)])
    run_command(["git", "-C", str(checkout), "fetch", "--quiet", "--depth", "1", JAVA_URL, JAVA_REVISION])
    run_command(["git", "-C", str(checkout), "checkout", "--quiet", "--detach", "FETCH_HEAD"])
    return checkout / "Code" / "src" / "heuristic"


def replace_once(text, before, after):
    if text.count(before) != 1:
        raise ValueError(f"Revisão Java incompatível no trecho {before!r}.")
    return text.replace(before, after, 1)


def append_members(text, members):
    end = text.rfind("}")
    if end == -1:
        raise ValueError("Classe Java incompleta.")
    return text[:end] + members + text[end:]


def patch_source(path, transform):
    path.write_text(transform(path.read_text(encoding="utf-8")), encoding="utf-8")


def patch_java_reference(source, mode, *, accelerated):
    ig = source / "algorithm" / "IG.java"
    local = source / "localSearch" / "LocalSearchEfficient_1_1.java"
    solution = source / "structure" / "Solution.java"
    gip = source / "constructive" / "GIP.java"

    patch_source(ig, lambda code: append_members(code, JAVA_FIXED_METHOD))
    # A variável era usada apenas em trechos comentados no original.
    patch_source(local, lambda code: replace_once(code, "        Solution lastSol=new Solution(instance);\n", ""))
    if mode == "original":
        return

    patch_source(ig, lambda code: replace_once(
        code, "        Collections.shuffle(selectedList,rnd);",
        "        Collections.sort(selectedList);\n        Collections.shuffle(selectedList,rnd);",
    ))

    def order_local_search(code):
        code = replace_once(
            code, "        Collections.shuffle(copySelected, RandomManager.getRandom());",
            "        Collections.sort(copySelected);\n        Collections.shuffle(copySelected, RandomManager.getRandom());",
        )
        return replace_once(code, "neighbours.iterator().next()", "Collections.min(neighbours)")

    patch_source(local, order_local_search)

    def order_solution(code):
        code = replace_once(
            code, "List<Integer> selectedList=new ArrayList<>(selectedNotSupportNodes);",
            "List<Integer> selectedList=new ArrayList<>(selectedNotSupportNodes);\n"
            "        Collections.sort(selectedList);",
        )
        return replace_once(
            code, "if(bestCount<count && !instance.getLeavesNodes().contains(i)){",
            "if((bestCount<count || (bestCount==count && i<bestNode))"
            " && !instance.getLeavesNodes().contains(i)){",
        )

    patch_source(solution, order_solution)
    if not accelerated:
        return

    # Mantém a regra de seleção e a poda, trocando as varreduras por estruturas incrementais.
    def add_caches(code):
        code = replace_once(code, "this.instance=instance;",
                            "this.instance=instance;\n        this.privateCounts=new int[instance.getNumNodes()];")
        code = replace_once(
            code, "List<Integer> selectedList=new ArrayList<>(selectedNotSupportNodes);",
            "List<Integer> selectedList=new ArrayList<>(redundant);",
        )
        code = replace_once(
            code, "selectedNotSupportNodes.add(node);",
            "selectedNotSupportNodes.add(node);\n            if (privateCounts[node] == 0) redundant.add(node);",
        )
        code = replace_once(
            code, "selectedNotSupportNodes.remove(node);",
            "selectedNotSupportNodes.remove(node);\n        redundant.remove(node);",
        )
        for method, next_method, implementation in (
            ("private void addWatcher", "public void removeNode", JAVA_ADD_WATCHER),
            ("private void removeWatcher", "public void copy", JAVA_REMOVE_WATCHER),
        ):
            start = code.index("    " + method)
            end = code.index("    " + next_method, start)
            code = code[:start] + implementation + code[end:]
        return append_members(code, JAVA_CACHE_MEMBERS)

    patch_source(solution, add_caches)
    for path in (ig, gip):
        patch_source(path, lambda code: replace_once(
            code, "solution.getBestNextNode()", "solution.getBestNextNodeWithQueue()"
        ))
    patch_source(local, lambda code: replace_once(
        code, "neighbours.retainAll(instance.getAdyacencyList()[neighbour]);",
        "List<Integer> row = instance.getAdyacencyList()[neighbour];\n"
        "                    neighbours.removeIf(v -> Collections.binarySearch(row, v) < 0);",
    ))


def compile_java_reference(repo, directory, mode, *, accelerated=True):
    if mode not in ("canonico", "original"):
        raise ValueError('Use JAVA_MODE = "canonico" ou "original".')
    upstream = java_checkout(repo, directory)
    if not (upstream / "algorithm" / "IG.java").is_file():
        raise FileNotFoundError(f"Classes heurísticas ausentes em {upstream}.")

    source = directory / "heuristic"
    shutil.copytree(upstream, source)
    patch_java_reference(source, mode, accelerated=accelerated)
    compiler = directory / "CompileIg.java"
    compiler.write_text(JAVA_COMPILER, encoding="utf-8")
    main = directory / "IgFixedMain.java"
    main.write_text(JAVA_MAIN, encoding="utf-8")
    classes = directory / "classes"
    classes.mkdir()
    files = [str(path) for path in sorted(source.rglob("*.java"))]
    run_command(["java", str(compiler), "-encoding", "UTF-8", "-d", str(classes), str(main), *files])
    return classes

## 3. Validar as soluções

Cada linha corresponde a um grafo e a um par `(semente, orçamento)`. Os IDs internos
seguem a ordem dos vértices de `G` nas duas versões e são convertidos de volta aos
rótulos originais antes de comparar. As referências são geradas na própria execução,
sem soluções pré-gravadas ou hashes específicos de uma instância.

- `valido_java` e `valido_python`: cada conjunto domina todos os vértices do grafo.
- `mesmo_tamanho`: as cardinalidades coincidem.
- `mesmo_conjunto`: todos os vértices coincidem.
- `diferenca_simetrica`: número de vértices que aparecem em apenas um dos conjuntos.
- `validacao_ok`: ambos os conjuntos são válidos e exatamente iguais.

Cardinalidades iguais, sozinhas, não aprovam a validação. Em caso de erro,
`validacao_ok` fica ausente e os campos `error_java`, `error_python` ou `error_graph`
explicam a falha. Uma divergência com execuções concluídas aparece como `False`.


In [4]:
def write_java_instance(G, path):
    if G.is_directed() or G.is_multigraph():
        raise ValueError("Use um grafo simples e não direcionado.")
    labels = tuple(G)
    positions = {node: pos for pos, node in enumerate(labels)}
    with path.open("w", encoding="ascii") as stream:
        stream.write(f"{len(labels)} {G.number_of_edges()}\n")
        for u, v in G.edges():
            if u == v:
                raise ValueError("Remova os laços antes de exportar a instância.")
            stream.write(f"{positions[u]} {positions[v]}\n")
    return labels


def run_java_fixed(classes, instance, labels, *, seed, iterations, beta):
    stdout = run_command([
        "java", "-cp", str(classes), "IgFixedMain",
        str(instance), str(seed), str(iterations), float(beta).hex(),
    ])
    lines = [line for line in stdout.splitlines() if line.startswith("IG_SOLUTION ")]
    if len(lines) != 1:
        raise ValueError("Saída Java sem uma linha única de solução.")
    cardinality, *vertices = map(int, lines[0].split()[1:])
    if cardinality != len(vertices) or any(not 0 <= v < len(labels) for v in vertices):
        raise ValueError("Saída Java com cardinalidade ou IDs inválidos.")
    solution = {labels[v] for v in vertices}
    if len(solution) != cardinality:
        raise ValueError("Saída Java com vértices repetidos.")
    return solution


def is_dominating_set(G, solution):
    return all(node in G for node in solution) and all(
        node in solution or any(neighbor in solution for neighbor in G[node]) for node in G
    )


def _fixed_cases(cases, beta):
    cases = tuple((index(seed), index(iterations)) for seed, iterations in cases)
    if any(not -(1 << 63) <= seed < (1 << 63) for seed, _ in cases):
        raise ValueError("A semente precisa caber em um long de Java (64 bits).")
    if any(not 0 <= iterations < (1 << 31) for _, iterations in cases):
        raise ValueError("Use um orçamento de iterações entre 0 e 2**31 - 1.")
    beta = float(beta)
    if not 0 <= beta <= 1:
        raise ValueError("beta deve estar entre 0 e 1.")
    return cases, beta


def _iter_fixed_results(G, name, classes, instance, cases, *, beta, continue_on_error):
    labels = write_java_instance(G, instance)
    for seed, iterations in cases:
        print(f"{name} | semente {seed} | limite {iterations} iterações", flush=True)
        row = {
            "grafo": name,
            "n_nodes": len(labels),
            "n_edges": G.number_of_edges(),
            "seed": seed,
            "max_iterations": iterations,
            "beta": beta,
            "modo_java": "canonico",
            "status_java": "error",
            "status_python": "error",
            "output_java": None,
            "output_python": None,
            "valido_java": None,
            "valido_python": None,
            "mesmo_tamanho": None,
            "mesmo_conjunto": None,
            "diferenca_simetrica": None,
            "validacao_ok": None,
            "error_java": None,
            "error_python": None,
            "error_graph": None,
        }
        solution_java = solution_py = None
        try:
            solution_java = run_java_fixed(
                classes, instance, labels, seed=seed, iterations=iterations, beta=beta
            )
            row.update(status_java="ok", output_java=len(solution_java),
                       valido_java=is_dominating_set(G, solution_java))
        except Exception as error:
            if not continue_on_error:
                raise
            row["error_java"] = f"{type(error).__name__}: {error}"

        try:
            solution_py = mds_solver_iterated_greedy(
                G, seed=seed, max_iterations=iterations, beta=beta
            )
            row.update(status_python="ok", output_python=len(solution_py),
                       valido_python=is_dominating_set(G, solution_py))
        except Exception as error:
            if not continue_on_error:
                raise
            row["error_python"] = f"{type(error).__name__}: {error}"

        if solution_java is not None and solution_py is not None:
            row.update(
                mesmo_tamanho=len(solution_java) == len(solution_py),
                mesmo_conjunto=solution_java == solution_py,
                diferenca_simetrica=len(solution_java ^ solution_py),
                validacao_ok=(row["valido_java"] and row["valido_python"]
                              and solution_java == solution_py),
            )
        yield row
        del solution_java, solution_py


def check_fixed_budgets_many(graph_paths, repo, cases, *, beta=0.2, continue_on_error=True):
    """Compila uma referência e valida os pares (semente, orçamento) em cada arquivo."""
    cases, beta = _fixed_cases(cases, beta)
    paths = tuple(Path(path) for path in graph_paths)
    rows = []
    if not paths or not cases:
        return pd.DataFrame(rows)

    with TemporaryDirectory(prefix="iterated-greedy-validacao-") as temporary:
        directory = Path(temporary)
        classes = compile_java_reference(Path(repo), directory, "canonico")
        instance = directory / "instance.txt"
        for graph_path in paths:
            G = None
            name = graph_path.name.removesuffix(".txt.gz")
            try:
                G = load_edgelist(graph_path)
                G = preprocess(G)
                rows.extend(_iter_fixed_results(
                    G, name, classes, instance, cases, beta=beta,
                    continue_on_error=continue_on_error,
                ))
            except Exception as error:
                if not continue_on_error:
                    raise
                message = f"{type(error).__name__}: {error}"
                print(f"{graph_path.name} | {message}", flush=True)
                for seed, iterations in cases:
                    rows.append({
                        "grafo": name, "seed": seed, "max_iterations": iterations,
                        "beta": beta, "modo_java": "canonico", "validacao_ok": None,
                        "mesmo_conjunto": None, "diferenca_simetrica": None,
                        "error_graph": message,
                    })
            finally:
                del G
                gc.collect()
    return pd.DataFrame(rows)


def check_fixed_budgets(graph_path, repo, cases, *, beta=0.2):
    """Valida um arquivo; falhas de execução interrompem a célula."""
    return check_fixed_budgets_many(
        (graph_path,), repo, cases, beta=beta, continue_on_error=False
    )


### Executar a validação

Com `RODAR_LOTE = False`, valida `INSTANCE_NAME`. Com `True`, percorre todos os
`*.txt.gz` de `RAW_DATA_DIR` com `iter_files`. A ordem `"size"` começa pelos menores
arquivos compactados; `"name"` usa ordem alfabética.

O lote compila a referência uma vez, libera cada grafo antes do próximo e continua
após falhas de uma execução. No modo individual, uma falha interrompe a célula.
Os resultados ficam em `resultados_iteracoes`.


In [7]:
TESTES_FIXOS = [(13, 100)]  # (semente, número de iterações).
RODAR_LOTE = False
ORDEM_ARQUIVOS = "size"  # "size" ou "name".

if TESTES_FIXOS:
    if RODAR_LOTE:
        graph_paths = iter_files(RAW_DATA_DIR, "*.txt.gz", order=ORDEM_ARQUIVOS)
        resultados_iteracoes = check_fixed_budgets_many(
            graph_paths, JAVA_REPO, TESTES_FIXOS, beta=BETA
        )
    else:
        GRAPH_PATH = Path(RAW_DATA_DIR) / f"{INSTANCE_NAME}.txt.gz"
        resultados_iteracoes = check_fixed_budgets(
            GRAPH_PATH, JAVA_REPO, TESTES_FIXOS, beta=BETA
        )
    display(resultados_iteracoes)


amazon0302 | semente 13 | limite 100 iterações


,grafo,n_nodes,n_edges,seed,max_iterations,beta,modo_java,status_java,status_python,output_java,output_python,valido_java,valido_python,mesmo_tamanho,mesmo_conjunto,diferenca_simetrica,validacao_ok,error_java,error_python,error_graph
0,amazon0302,262111,899792,13,100,0.2,canonico,ok,ok,36711,36711,True,True,True,True,0,True,None,None,None


### Conferir o resultado

Cada caso aprovado deve apresentar `valido_java=True`, `valido_python=True`,
`mesmo_conjunto=True`, `diferenca_simetrica=0` e `validacao_ok=True`.
O resumo distingue uma solução diferente ou inválida de um teste que não pôde
ser concluído por erro. Amplie os grafos, sementes e orçamentos para cobrir mais casos.


In [6]:
if TESTES_FIXOS and not resultados_iteracoes.empty:
    aprovados = resultados_iteracoes["validacao_ok"].eq(True)
    sem_conclusao = resultados_iteracoes["validacao_ok"].isna()
    divergencias = ~aprovados & ~sem_conclusao
    print(
        f"Aprovados: {int(aprovados.sum())} | "
        f"Divergentes ou inválidos: {int(divergencias.sum())} | "
        f"Sem conclusão por erro: {int(sem_conclusao.sum())}"
    )
    if (~aprovados).any():
        display(resultados_iteracoes.loc[~aprovados])


Aprovados: 1 | Divergentes ou inválidos: 0 | Sem conclusão por erro: 0


## 4. Usar o algoritmo em outros experimentos

O módulo Python continua independente da referência Java e deste notebook:

```python
from complex_networks.algorithms.mds.iterated_greedy import mds_solver_iterated_greedy

solucao = mds_solver_iterated_greedy(G, seed=13, max_iterations=200, beta=0.2)
```

`max_iterations` permanece um parâmetro da implementação que você recebeu.
Neste notebook ele alinha as execuções para validar os conjuntos. Definir um
protocolo de comparação entre algoritmos é uma etapa separada.
